In [ ]:
import re
import json
from pathlib import Path
import pandas as pd
import torch
import sys
from Models.model import GLSTMBatch  # ajuste o import se necessário

sys.path.append("../src")

from Models.model import GLSTMBatch
from Graph.graph_related_utils import knn_topology, distance_topology, adjacency_matrix

In [3]:


def carregar_runs_modelos(
    root_dir=r"C:\Experiments\multi_step_12_03_06_25_learn_v1\WINDOW_30_HORIZON_5",
    load_state_dict=True,
    load_hist=False,
):
    root = Path(root_dir)
    rows = []

    for summary_path in root.rglob("run_summary.json"):
        run_dir = summary_path.parent
        rel = run_dir.relative_to(root)
        parts = list(rel.parts)

        window_tag = next((p for p in parts if p.upper().startswith("WINDOW_")), None)
        exp_id = next((p for p in parts if re.match(r"^exp[_-]?\d+$", p, flags=re.IGNORECASE)), None)

        m_batch = re.search(r"BATCH_TEST_(\d+)", run_dir.name, flags=re.IGNORECASE)
        seed = int(m_batch.group(1)) if m_batch else None

        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)

        row = {
            "summary_path": str(summary_path),
            "run_dir": str(run_dir),
            "window_tag": window_tag,
            "exp_id": exp_id,
            "seed": seed,
            **summary,  # inclui métricas e hiperparâmetros do json
        }

        if load_state_dict:
            state_path = run_dir / "model_state_dict.pt"
            if state_path.exists():
                row["model_state_dict"] = torch.load(state_path, map_location="cpu")

        if load_hist:
            hist_path = run_dir / "hist.pt"
            if hist_path.exists():
                row["history"] = torch.load(hist_path, map_location="cpu")

        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        print("Nenhuma run encontrada.")
        return df
    return df.sort_values(["exp_id", "seed"], na_position="last").reset_index(drop=True)



def load_all_models(
    root_dir,
    N,
    edge_index,
    in_channels,
    out_channels,
    lstm_layers=1,
    learn_adj=True,
    device="cpu",
    return_state_dict_only=False,
):
    root = Path(root_dir)
    models = []

    for summary_path in root.rglob("run_summary.json"):
        run_dir = summary_path.parent
        state_path = run_dir / "model_state_dict.pt"
        if not state_path.exists():
            continue

        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)

        hidden_dim = summary.get("hidden_dim", summary.get("hiddem_dim"))
        if hidden_dim is None:
            continue

        state = torch.load(state_path, map_location=device)

        if return_state_dict_only:
            models.append((run_dir, summary, state))
            continue

        model = GLSTMBatch(
            N=N,
            edge_index=edge_index.to(device),
            in_channels=in_channels,
            hidden_size=hidden_dim,
            out_channels=out_channels,
            lstm_layers=lstm_layers,
            learn_adj=learn_adj,
        ).to(device)

        model.load_state_dict(state)
        model.eval()
        models.append((run_dir, summary, model))

    return models


In [10]:
runs_df = carregar_runs_modelos(load_state_dict=False, load_hist=True)
tensor_test = torch.tensor(runs_df.hiddem_dim.to_numpy(), dtype=torch.int)

In [11]:
print(tensor_test)

tensor([ 32,  64, 128,  32,  64, 128,  32,  64, 128], dtype=torch.int32)


In [ ]:
root = r"C:\Experiments\multi_step_12_03_06_25_learn_v1\WINDOW_30_HORIZON_5"
models = load_all_models(
    root_dir=root,
    N=N,
    edge_index=edge_index,
    in_channels=32,
    out_channels=horizon,
    lstm_layers=1,
    learn_adj=True,
    device=device,
)
print(len(models))

,summary_path,run_dir,window_tag,exp_id,seed,hiddem_dim,train_period,device,epochs_requested,epochs_ran,...,weight_decay,patience,batch_size,best_val_loss,last_val_mse,last_val_mae,last_val_mape,last_val_r2,total_time_s,history
0,C:\Experiments\multi_step_12_03_06_25_learn_v1...,C:\Experiments\multi_step_12_03_06_25_learn_v1...,None,exp_002,0,32,30,cuda,350,61,...,0.001,35,64,0.875923,0.878209,0.765517,115.991705,0.014978,553.900984,"{'train_loss': [0.9872363917529583, 0.95588819..."
1,C:\Experiments\multi_step_12_03_06_25_learn_v1...,C:\Experiments\multi_step_12_03_06_25_learn_v1...,None,exp_003,0,64,30,cuda,350,89,...,0.001,35,64,0.876227,0.878659,0.769593,119.192915,0.013282,706.326538,"{'train_loss': [0.992275508493185, 0.949468051..."
2,C:\Experiments\multi_step_12_03_06_25_learn_v1...,C:\Experiments\multi_step_12_03_06_25_learn_v1...,None,exp_004,0,128,30,cuda,350,64,...,0.001,35,64,0.876619,0.878258,0.771056,120.130861,0.013510,568.643290,"{'train_loss': [1.0127781242132188, 0.95108420..."
3,C:\Experiments\multi_step_12_03_06_25_learn_v1...,C:\Experiments\multi_step_12_03_06_25_learn_v1...,None,exp_005,0,32,30,cuda,350,83,...,0.001,35,64,0.864706,0.868941,0.751279,128.062532,0.026347,688.599201,"{'train_loss': [1.0772507280111312, 1.01501557..."
4,C:\Experiments\multi_step_12_03_06_25_learn_v1...,C:\Experiments\multi_step_12_03_06_25_learn_v1...,None,exp_006,0,64,30,cuda,350,91,...,0.001,35,64,0.866992,0.878844,0.761224,133.359921,0.010885,724.865947,"{'train_loss': [1.0374078392982482, 0.99573175..."
